# DG-3 walkthrough — cross-method comparison and the failure-asymmetry-clearance gap

**Decision Gate:** DG-3 (cross-method validation; ≥2 methods with non-overlapping failure modes).
**Verdict:** RUNNER-COMPLETE; failure-asymmetry-cleared PASS still pending.
**Cards exercised:** [C1](../benchmarks/benchmark_cards/C1_cross-method-pure-dephasing_v0.1.0.yaml), [C2](../benchmarks/benchmark_cards/C2_cross-method-spin-boson_v0.1.0.yaml).
**Authoritative status:** [`docs/validity_envelope.md`](../docs/validity_envelope.md).

> **What this notebook is.** This is *not* a demonstration of a passed Decision Gate. DG-3 cards C1 and C2 are reachable via the runner's cross-method branch (`reporting.benchmark_card._run_cross_method`) and produce a structured verdict, but at the present truncation both fixtures FAIL convergence: `exact_finite_env` (finite-system class) and `qutip_reference` (solver-default class) disagree by ~10⁻¹ on the reduced density matrix, vastly exceeding the frozen `1.0e-6` agreement threshold.
>
> The point of the walkthrough is to demonstrate what an honest carve-out looks like: the runner is reachable, the verdict is auditable, and the gap to a failure-asymmetry-cleared PASS is documented (it requires either bath-convergence under the cleared registry or a third reference method from a non-overlapping failure-mode class — HEOM, TEMPO, MCTDH, or pseudomode/chain-mapping per Sail v0.5 §5 Tier 3 and [`docs/benchmark_protocol.md`](../docs/benchmark_protocol.md) §2).

## DG-3 readiness vs. failure-asymmetry clearance (Sail v0.5 §9)

Per Sail v0.5 §9, two distinct properties are tracked separately for any DG-3-relevant method pair:

- **Implementation readiness** — the methods are implemented, callable, and produce output for the model under test.
- **Failure-asymmetry clearance** — the pair satisfies the §2 failure-mode asymmetry rule (the two methods must come from different failure-mode classes).

The current pair `exact_finite_env` (`finite-system` class) + `qutip_reference` (`solver-default` class) satisfies the failure-asymmetry rule structurally (the two classes are distinct), but the cards FAIL convergence at the frozen finite-bath truncation, so the *failure-asymmetry-cleared PASS* requires further work.

## 0. Setup

In [ ]:
import copy
from pathlib import Path

import yaml

import cbg
from reporting import benchmark_card as bc

print(f"cbg.__version__       = {cbg.__version__}")
print(f"cbg.__sail_version__  = {cbg.__sail_version__}")
print(f"cbg.__ledger_anchor__ = {cbg.__ledger_anchor__}")

## 1. Load card C2 — cross-method, σ_x coupling

Card C2 v0.1.0 (DG-3, spin-boson σ_x model) compares two reference methods drawn from non-overlapping failure-mode classes:

- `exact_finite_env` — finite-environment exact diagonalisation (failure mode: finite-bath truncation, recurrence times).
- `qutip_reference` — QuTiP master-equation solver (failure mode: solver-default class; Born–Markov / secular assumptions baked into solver choice).

The card has two test cases: a thermal bath and a coherent-displaced bath (profile `delta-omega_c`).

In [2]:
C2_PATH = Path(cbg.__file__).parent.parent / "benchmarks" / "benchmark_cards" / "C2_cross-method-spin-boson_v0.1.0.yaml"
card = bc.load_card(C2_PATH)
print(f"card_id            = {card.card_id} {card.version}")
print(f"dg_target          = {card.dg_target}")
print(f"model              = {card.model}")
print(f"target_observable  = {card.frozen_parameters['comparison']['target_observable']}")
print(f"error_metric       = {card.frozen_parameters['comparison']['error_metric']}")
print(f"agreement threshold = {card.threshold}")
print()
print("test cases:")
for case in card.frozen_parameters["model"]["test_cases"]:
    bath = case["bath_state"]
    label = bath["family"]
    if label == "coherent_displaced":
        label += f" (profile={bath['displacement_profile']!r})"
    print(f"  • {case['name']}  —  {label}")

card_id            = C2 v0.1.0
dg_target          = DG-3
model              = spin_boson_sigma_x
target_observable  = reduced_density_matrix
error_metric       = inter_method_relative_frobenius
agreement threshold = 1e-06

test cases:
  • thermal_bath_cross_method  —  thermal
  • displaced_bath_delta_omega_c_cross_method  —  coherent_displaced (profile='delta-omega_c')


## 2. Run the card via `run_card`

We use a reduced time grid (`t_end=5.0`, `n_points=21`) so the QuTiP reference and the finite-environment reference complete in seconds. The full frozen card uses `t_end=20.0`, `n_points=200`. Both reduced and full runs return the same verdict (FAIL): the methods agree on the reduced density-matrix trajectory only at the ~10⁻¹ level under finite-bath truncation, far above the frozen `1.0e-6` threshold.

In [3]:
data = copy.deepcopy(yaml.safe_load(open(C2_PATH)))
data["frozen_parameters"]["numerical"]["time_grid"]["t_end"] = 5.0
data["frozen_parameters"]["numerical"]["time_grid"]["n_points"] = 21
bc.validate_card_data(data)
reduced_card = bc._data_to_card(data)

result = bc.run_card(reduced_card)
print(f"verdict          = {result.verdict}")
print(f"runner_version   = {result.runner_version}")
print(f"# test_cases     = {len(result.test_case_results)}")
print()
print("per-case agreement (inter-method relative Frobenius):")
print(f"{'name':<48} {'error':>10}   {'threshold':>10}   passed")
print("-" * 90)
for tcr in result.test_case_results:
    print(f"{tcr.name:<48} {tcr.error:>10.3e}   {tcr.threshold:>10.0e}   {tcr.passed}")

/Users/uwarring/Documents/GitHub/oqs-cbg-pipeline/.venv/lib/python3.13/site-packages/qutip/__init__.py:24: UserWarning: matplotlib not found: Graphics will not work.
  warnings.warn("matplotlib not found: Graphics will not work.")


verdict          = FAIL
runner_version   = 0.1.0
# test_cases     = 2

per-case agreement (inter-method relative Frobenius):
name                                                  error    threshold   passed
------------------------------------------------------------------------------------------
thermal_bath_cross_method                         5.383e-01        1e-06   False
displaced_bath_delta_omega_c_cross_method         5.265e-01        1e-06   False


/Users/uwarring/Documents/GitHub/oqs-cbg-pipeline/.venv/lib/python3.13/site-packages/qutip/core/coefficient.py:424: UserWarning: `cython`, `setuptools` and `filelock` are required for compilation of string coefficents. Falling back on `eval`.
  warnings.warn(


## 3. Reading the FAIL — what does the disagreement mean?

Both methods *run cleanly*: the runner returns a structured `FAIL` verdict with finite per-case errors. The disagreement is not a bug; it reflects the *failure modes that we expect to differ*:

- **`exact_finite_env`** truncates the continuous bath to a finite mode set (`bath_mode_cutoff = 1024` modes for the full card; `2 × 2 = 4` for the reduced fixture used here would give an even cruder approximation). Any disagreement at finite truncation is an honest finite-system-class artefact.
- **`qutip_reference`** uses the solver's default master-equation assumptions (Lindblad / secular). The default-class assumptions can systematically deviate from the exact non-Markovian reduced dynamics in regimes where the bath retains memory.

A failure-asymmetry-cleared DG-3 PASS requires either:

1. **Convergence under the cleared registry**: increasing the finite-environment truncation parameters (more bath modes, more levels) until both methods agree at the frozen threshold. If they fail to agree even at large truncation, the disagreement is genuinely between method classes, not just an under-resolved benchmark.
2. **A third reference method from a non-overlapping failure-mode class.** HEOM (hierarchy-truncation class), TEMPO / process tensor (memory-cutoff class), MCTDH (basis-truncation class), or pseudomode / chain-mapping (auxiliary-system class) per [`docs/benchmark_protocol.md`](../docs/benchmark_protocol.md) §2. A clean three-method agreement would satisfy the failure-asymmetry rule even if the present pair never converges to threshold.

## 4. The pure-dephasing sibling — Card C1

Card C1 v0.1.0 (DG-3, pure-dephasing model) is the sigma_z-coupling sibling of C2. Both fixtures (thermal and `delta-omega_c` displaced) similarly run to a clean `FAIL` verdict at the frozen threshold. The full frozen run is in the test suite; we report the verdict directly here from a smaller smoke run.

In [4]:
C1_PATH = Path(cbg.__file__).parent.parent / "benchmarks" / "benchmark_cards" / "C1_cross-method-pure-dephasing_v0.1.0.yaml"
data = copy.deepcopy(yaml.safe_load(open(C1_PATH)))
data["frozen_parameters"]["numerical"]["time_grid"]["t_end"] = 5.0
data["frozen_parameters"]["numerical"]["time_grid"]["n_points"] = 21
bc.validate_card_data(data)
reduced_card_c1 = bc._data_to_card(data)

result_c1 = bc.run_card(reduced_card_c1)
print(f"C1 verdict       = {result_c1.verdict}")
for tcr in result_c1.test_case_results:
    print(f"  {tcr.name:<48} error={tcr.error:.3e}  passed={tcr.passed}")

C1 verdict       = FAIL
  thermal_bath_cross_method                        error=2.928e-01  passed=False
  displaced_bath_delta_omega_c_cross_method        error=3.089e-01  passed=False


## 5. Outcome

The DG-3 cross-method runner is reachable on all four C1+C2 fixtures (pure-dephasing thermal, pure-dephasing displaced `delta-omega_c`, σ_x thermal, σ_x displaced `delta-omega_c`). Both cards return a structured `FAIL` verdict at the frozen `1.0e-6` agreement threshold, with disagreement at the ~10⁻¹ level under finite-bath truncation.

**What this authorises and does not authorise.**

- **Authorises** citing the repository for the cross-method *implementation-readiness* level: the runner is reachable, both methods complete, the verdict is auditable, and the failure-mode classes of the two methods are distinct (`finite-system` vs `solver-default`).
- **Does not authorise** citing the repository for cross-method *failure-asymmetry-cleared* validation. The pair currently FAILs convergence under the frozen truncation, and a single failing pair from two distinct method classes does not prove non-overlapping failure modes — only that the two methods disagree, which is also consistent with truncation artefacts.

> **Citation note.** Per Sail v0.5 §9, DG-3 reports must explicitly state which level of pass is being claimed. Citing C1/C2 results requires naming both the level (implementation-readiness vs failure-asymmetry-cleared) and, for displaced-bath cases, the registered displacement profile.

The path forward is recorded in the [validity envelope](../docs/validity_envelope.md) DG-3 row: either bath-convergence under the cleared registry, or a third reference method from a non-overlapping failure-mode class.